Comparing the diversity of narrative reports for reports with the same event tags to assess inter-rater reliability

In [1]:
import pandas as pd
import numpy as np

from datasets import load_dataset

from huggingface_hub import hf_hub_download

import utils 

/home/maciej/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
events_df = pd.read_csv("../data-processing/events.csv")
events_df.rename(columns={"Unnamed: 0" : "acn"}, inplace=True)
#events_df.set_index("acn", inplace=True)

combined_events_df = events_df[events_df.columns[:1]]

# Remove NaN instances from set
combined_events_df["temp"] = events_df[events_df.columns[2:]].agg(set, axis=1)
for index, row in combined_events_df.iterrows():
    no_nan_list = []
    for event in row["temp"]: 
        if isinstance(event, str): no_nan_list.append(event)
    combined_events_df.at[index, "temp"] = no_nan_list

combined_events_df

# Converted to a string because strings can be hashed, which is useful for finding duplicates
combined_events_df["events"] = [", ".join(r["temp"]) for i, r in combined_events_df.iterrows()]
combined_events_df.drop(columns="temp", inplace=True)
combined_events_df

/tmp/ipykernel_151684/3739016954.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_events_df["temp"] = events_df[events_df.columns[2:]].agg(set, axis=1)
/tmp/ipykernel_151684/3739016954.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_events_df["events"] = [", ".join(r["temp"]) for i, r in combined_events_df.iterrows()]
/tmp/ipykernel_151684/3739016954.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentati

,acn,events
0,2260174,anomaly_inflight_event_/_encounter_unstabilize...
1,2260249,"when_detected_in_flight, anomaly_aircraft_equi..."
2,2260370,"when_detected_in_flight, anomaly_aircraft_equi..."
3,2261277,"when_detected_in_flight, anomaly_deviation_/_d..."
4,2261317,"when_detected_in_flight, detector_automation_a..."
...,...,...
24529,2314733,"anomaly_aircraft_equipment_problem_critical, a..."
24530,2314971,"result_general_police_/_security_involved, ano..."
24531,2314972,anomaly_deviation_/_discrepancy_-_procedural_f...
24532,2314978,"when_detected_in_flight, detector_person_air_t..."


In [3]:
narratives_df = pd.read_csv("../data-processing/narratives.csv")
narratives_df.rename(columns={"Unnamed: 0" : "acn"}, inplace=True)
narratives_df.set_index("acn", inplace=True)

narratives_df.head()

,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative
acn,,,
2260174,ambiguous,Aircraft vectored in within 1NM to final appro...,NaN
2260249,ambiguous,While on short final we received a glideslope ...,NaN
2260370,aircraft,Flying at cruise; FL350; the FO was the PF and...,At cruise; FL350; during level-off; the Captai...
2261277,humanfactors,On Day 0 around XA:30; I forgot to get LAANC a...,NaN
2261317,procedure,Divert into ZZZ from ZZZ1. FO flying. Vectors ...,Extremely strong winds blew us off the LOC whi...


In [4]:
# Group by the same events string, and then turn those ACNs into lists
duplicates_series = combined_events_df.groupby("events")["acn"].apply(list)
duplicates_series

for index, acns in duplicates_series.items():
    if len(acns) <= 1:          # Not a duplicate
        duplicates_series.drop(index, inplace=True)

duplicates_series

events
anomaly_aircraft_equipment_problem_critical                                                                                                                                                                                                                                                                                                               [2032791, 2260241, 2261197, 2264946, 2270182, ...
anomaly_aircraft_equipment_problem_critical, anomaly_deviation_/_discrepancy_-_procedural_clearance                                                                                                                                                                                                                                                           [2285608, 2296521, 2298943, 2304776, 2314733]
anomaly_aircraft_equipment_problem_critical, anomaly_deviation_/_discrepancy_-_procedural_clearance, anomaly_deviation_/_discrepancy_-_procedural_published_material_/_policy                            

### Analysis

In [ ]:
def printDuplicateNarratives(index : int):
    print("Event Tags:")
    tags = duplicates_series.index[index].split(", ")
    
    for t in tags:
        print('\t', t)

    print()

    for i in range(len(duplicates_series.iloc[index])):
        print(f"===== Narrative {i} =====")
        print(narratives_df.at[duplicates_series.iloc[index][i], "Report_1_Narrative"])
        print()

In [ ]:
# Note: if you're using VS Code like I am, you can turn on word wrapping in settings under notebook.output.wordWrap

printDuplicateNarratives(10)

Event Tags:
	 anomaly_aircraft_equipment_problem_critical
	 anomaly_flight_deck_/_cabin_/_aircraft_event_illness_/_injury
	 detector_person_flight_attendant
	 detector_person_flight_crew
	 result_general_work_refused
	 result_general_flight_cancelled_/_delayed
	 when_detected_aircraft_in_service_at_gate
	 anomaly_flight_deck_/_cabin_/_aircraft_event_smoke_/_fire_/_fumes_/_odor

===== Narrative 0 =====
During the boarding process; a rotten egg smell was detected in the boarding area; front galley; and main cabin. Smell was reported to the captain; and he determined that we were experiencing an odor event. Captain came on the PA and instructed everyone to deplane ASAP. Crew was then met by ops who instructed us to start a report; call out sick and receive medical attention asap. We took a ride hailing service to an ER had blood drawn and received oxygen. All 4 of us flight attendants were declared good to go as our blood work came back with good readings. We were released and returned to